# Respondents Detection


In [ ]:
import sys
import os
from pathlib import Path

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root (2 levels up from src/python)
current_dir = Path(os.getcwd())
repo_root = current_dir.parents[1]

# Add src/python to path for local imports
src_python = repo_root / 'src' / 'python'
if str(src_python) not in sys.path:
    sys.path.insert(0, str(src_python))

from utils import read_clean_sheds
from outliers_functions import (
    run_outlier_detection,
    plot_completion_distribution,
    plot_waves_ridges,
    analyze_all_straightliners,
)

# Load config
with open(repo_root / 'config.yaml', 'r') as f:
    config = yaml.safe_load(f)

data_dir = Path(config['paths']['data_dir'])

years = [2016, 2017, 2018, 2019, 2020, 2021, 2023, 2025]

waves = {}
for year in years:
    file_path = data_dir / config['sheds_files'][str(year)]
    wave_name = str(year)
    df, _meta = read_clean_sheds(str(file_path))
    waves[wave_name] = df

# All unique IDs across waves
all_ids = pd.concat(
    [df[['id']] for df in waves.values()],
    ignore_index=True
)['id'].unique()

print(f'\nTotal unique IDs across all waves: {len(all_ids)}')

print('\n=== Summary ===')
total_respondents = sum(len(df) for df in waves.values())
print(f'Total respondents (all waves): {total_respondents}')
print(f'Total unique IDs: {len(all_ids)}')
print(f'Duplicate appearances: {total_respondents - len(all_ids)}')

id_counts = (
    pd.concat([df[['id']] for df in waves.values()], ignore_index=True)
    ['id']
    .value_counts()
    .reset_index()
)
id_counts.columns = ['id', 'n']

print('\nDistribution of appearances:')
print(id_counts['n'].value_counts().sort_index())

## Data Filtering Config


In [ ]:
# Duration filtering (in minutes)
MIN_DURATION = 0
MAX_DURATION = 100

waves_filtered = {}
for wave_name, df in waves.items():
    mask = (
        df['q_totalduration'].notna() &
        (df['q_totalduration'] >= MIN_DURATION) &
        (df['q_totalduration'] <= MAX_DURATION)
    )
    waves_filtered[wave_name] = df[mask].copy()

print('\n=== Filtering Summary ===')
print(f'Duration range: {MIN_DURATION} to {MAX_DURATION} minutes\n')

filtering_summary = pd.DataFrame({
    'Wave': list(waves.keys()),
    'Original': [len(waves[w]) for w in waves],
    'Filtered': [len(waves_filtered[w]) for w in waves_filtered],
})
filtering_summary['Removed'] = filtering_summary['Original'] - filtering_summary['Filtered']
filtering_summary['Pct_Kept'] = (filtering_summary['Filtered'] / filtering_summary['Original'] * 100).round(1)

print(filtering_summary.to_string(index=False))

## Run Outlier Detection


In [ ]:
wave_summary_rows = []
wave_results = {}
plots = {}

for wave_name in waves_filtered:
    print(f'WAVE: {wave_name}')

    data_original = waves[wave_name]
    below_10 = int((data_original['q_totalduration'] < 10).sum())
    above_60 = int((data_original['q_totalduration'] > 60).sum())
    outside_range = below_10 + above_60

    data_filtered = waves_filtered[wave_name]
    total = len(data_filtered)
    total_original = len(data_original)

    median_duration = data_filtered['q_totalduration'].median()

    results = run_outlier_detection(data_filtered)

    # Inconsistency type breakdown
    incons_detail = results.loc[results['inconsistent'] == True, 'inconsistency_types']
    if len(incons_detail) > 0:
        print('\n--- Inconsistency type breakdown ---')
        all_types = []
        for s in incons_detail:
            all_types.extend([t.strip() for t in s.split(';')])
        type_counts = pd.Series(all_types).value_counts()
        for itype, cnt in type_counts.items():
            print(f'  {itype:<40} {cnt} ({100 * cnt / len(incons_detail):.1f}% of inconsistent)')
        n_multi = incons_detail.str.contains(';').sum()
        print(f'\n  Respondents with >1 inconsistency type: {n_multi}\n')

    wave_results[wave_name] = results

    timing_speeders = int(results.get('timing_speeder', pd.Series(dtype=bool)).sum())
    straightliners = int(results.get('straightline', pd.Series(dtype=bool)).sum())
    inconsistent = int(results.get('inconsistent', pd.Series(dtype=bool)).sum())
    high_risk_count = int((results['risk_score'] >= 2).sum())
    avg_risk = results['risk_score'].mean()

    sl_stats = results.attrs.get('straightline_summary_stats', {})
    n_got_questions = sl_stats.get('n_got_questions', None)

    print(f'Total (original): {total_original}')
    print(f'Total (filtered): {total}')
    print(f'Below 10 min: {below_10} ({100 * below_10 / total_original:.1f}%)')
    print(f'Above 60 min: {above_60} ({100 * above_60 / total_original:.1f}%)')
    print(f'Outside 10-60 min range: {outside_range} ({100 * outside_range / total_original:.1f}%)')
    print(f'5th percentile: {timing_speeders} ({100 * timing_speeders / total:.1f}%)')
    print(f'Straightliners: {straightliners} ({100 * straightliners / total:.1f}%)')
    if n_got_questions is not None:
        print(f'  • Got >2 questions: {n_got_questions}')
        pct_of_got = 100 * straightliners / n_got_questions if n_got_questions > 0 else 0
        print(f'  • Straightliners as % of those who got questions: {pct_of_got:.1f}%')
    print(f'Inconsistent: {inconsistent} ({100 * inconsistent / total:.1f}%)')
    print(f'High-risk: {high_risk_count} ({100 * high_risk_count / total:.2f}%)\n')

    plots[wave_name] = plot_completion_distribution(data_filtered, wave_name, show_plot=False)
    plt.close()

    got_q = n_got_questions if n_got_questions is not None else total
    sl_pct_got = (100 * straightliners / got_q) if got_q > 0 else 0

    wave_summary_rows.append({
        'Wave': wave_name,
        'Total_Respondents': total,
        'Median_Duration': median_duration,
        'Below_10_min': below_10,
        'Below_10_Pct': (below_10 / total_original) * 100,
        'Above_60_min': above_60,
        'Above_60_Pct': (above_60 / total_original) * 100,
        'Outside_10_60_min': outside_range,
        'Outside_10_60_Pct': (outside_range / total_original) * 100,
        'Stability': timing_speeders,
        'Got_Questions': got_q,
        'Straightliners': straightliners,
        'Straightliners_Pct_Total': (straightliners / total) * 100,
        'Straightliners_Pct_Got': sl_pct_got,
        'Inconsistent': inconsistent,
        'High_Risk_Count': high_risk_count,
        'High_Risk_Pct': (high_risk_count / total) * 100,
        'Avg_Risk_Score': avg_risk,
    })

wave_summary = pd.DataFrame(wave_summary_rows)

print('\n=== WAVE SUMMARY TABLE ===')
print(wave_summary.to_string(index=False))

print('\n=== DURATION FILTERING SUMMARY ===')
duration_table = wave_summary[[
    'Wave', 'Total_Respondents',
    'Below_10_min', 'Below_10_Pct',
    'Above_60_min', 'Above_60_Pct',
    'Outside_10_60_min', 'Outside_10_60_Pct'
]].copy()
duration_table['Below_10_Pct'] = duration_table['Below_10_Pct'].apply(lambda x: f'{x:.1f}%')
duration_table['Above_60_Pct'] = duration_table['Above_60_Pct'].apply(lambda x: f'{x:.1f}%')
duration_table['Outside_10_60_Pct'] = duration_table['Outside_10_60_Pct'].apply(lambda x: f'{x:.1f}%')
print(duration_table.to_string(index=False))

print('\n=== STRAIGHTLINING SUMMARY ===')
straightlining_table = wave_summary[[
    'Wave', 'Total_Respondents',
    'Got_Questions', 'Straightliners',
    'Straightliners_Pct_Total', 'Straightliners_Pct_Got'
]].copy()
straightlining_table['Straightliners_Pct_Total'] = straightlining_table['Straightliners_Pct_Total'].apply(lambda x: f'{x:.1f}%')
straightlining_table['Straightliners_Pct_Got'] = straightlining_table['Straightliners_Pct_Got'].apply(lambda x: f'{x:.1f}%')
print(straightlining_table.to_string(index=False))

## Completion Time Distribution Plots


In [ ]:
n_waves = len(plots)
ncols = 2
nrows = (n_waves + 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 5))
axes_flat = axes.flatten() if n_waves > 1 else [axes]

for ax in axes_flat:
    ax.set_visible(False)

for idx, (wave_name, wave_fig) in enumerate(plots.items()):
    # Re-draw each wave figure into the subplot grid
    ax_src = wave_fig.axes[0]
    axes_flat[idx].set_visible(True)
    # Copy title/labels from original figure
    axes_flat[idx].set_title(ax_src.get_title(), fontweight='bold', fontsize=11)
    axes_flat[idx].set_xlabel(ax_src.get_xlabel())
    axes_flat[idx].set_ylabel(ax_src.get_ylabel())

    data_filtered = waves_filtered[wave_name]
    duration = data_filtered['q_totalduration'].dropna()
    fast_threshold = duration.quantile(0.05)
    mean_time = duration.mean()

    axes_flat[idx].hist(duration, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    axes_flat[idx].axvspan(duration.min(), fast_threshold, alpha=0.2, color='red', zorder=0)
    axes_flat[idx].axvline(10, color='gray', linestyle='dotted', linewidth=1)
    axes_flat[idx].axvline(60, color='gray', linestyle='dotted', linewidth=1)
    axes_flat[idx].axvline(mean_time, color='darkblue', linestyle='solid', linewidth=1)
    axes_flat[idx].set_xlim(10, 100)
    axes_flat[idx].set_xticks(range(10, 101, 10))
    axes_flat[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Ridge Plot of Completion Times Across Waves


In [ ]:
fig_ridges = plot_waves_ridges(waves_filtered)
plt.show()

## Violin + Box Plot of Completion Times Across Waves


In [ ]:
combined = pd.concat(
    [
        waves_filtered[wave].assign(Wave=wave)[['Wave', 'q_totalduration']].rename(columns={'q_totalduration': 'Duration'})
        for wave in waves_filtered
    ],
    ignore_index=True
)

wave_labels = sorted(combined['Wave'].unique())
data_by_wave = [combined.loc[combined['Wave'] == w, 'Duration'].dropna().values for w in wave_labels]

fig, ax = plt.subplots(figsize=(14, 7))

positions = range(1, len(wave_labels) + 1)
colors = plt.cm.Set2(np.linspace(0, 1, len(wave_labels)))

vp = ax.violinplot(data_by_wave, positions=list(positions), showmedians=False, showextrema=False)
for pc, color in zip(vp['bodies'], colors):
    pc.set_facecolor(color)
    pc.set_alpha(0.6)

bp = ax.boxplot(data_by_wave, positions=list(positions), widths=0.2,
                patch_artist=True, showfliers=True,
                flierprops=dict(marker='o', alpha=0.3, markersize=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.axhline(10, linestyle='dotted', color='gray', alpha=0.7)
ax.axhline(60, linestyle='dotted', color='gray', alpha=0.7)

ax.set_xticks(list(positions))
ax.set_xticklabels(wave_labels, fontsize=14, fontweight='bold')
ax.set_xlabel('Wave', fontsize=16, fontweight='bold')
ax.set_ylabel('Completion Time (minutes)', fontsize=16, fontweight='bold')

max_dur = combined['Duration'].max(skipna=True)
ax.set_yticks(range(0, int(max_dur) + 10, 5))
ax.grid(True, axis='y', alpha=0.3)
ax.legend_.remove() if ax.get_legend() else None

plt.tight_layout()
plt.show()

## Investigate Straightliners


In [ ]:
straightliner_details = analyze_all_straightliners(waves_filtered)

# Examine the most recent available wave
most_recent = max(straightliner_details.keys()) if straightliner_details else None
if most_recent:
    print(f'\nStraightliners for wave {most_recent}:')
    print(straightliner_details[most_recent])

## Summary of Flagged Respondents


In [ ]:
print('=== FLAGGED RESPONDENTS SUMMARY ===')
for wave_name, results in wave_results.items():
    total = len(results)
    timing_speeders = int(results.get('timing_speeder', pd.Series(dtype=bool)).sum())
    straightliners = int(results.get('straightline', pd.Series(dtype=bool)).sum())
    inconsistent = int(results.get('inconsistent', pd.Series(dtype=bool)).sum())
    high_risk = int((results['risk_score'] >= 2).sum())

    print(f'\nWave {wave_name} (n={total}):')
    print(f'  Timing speeders : {timing_speeders:4d} ({100 * timing_speeders / total:.1f}%)')
    print(f'  Straightliners  : {straightliners:4d} ({100 * straightliners / total:.1f}%)')
    print(f'  Inconsistent    : {inconsistent:4d} ({100 * inconsistent / total:.1f}%)')
    print(f'  High-risk (>=2) : {high_risk:4d} ({100 * high_risk / total:.2f}%)')

    flag_vars = [v for v in ['timing_speeder', 'straightline', 'inconsistent'] if v in results.columns]
    if flag_vars:
        print(f'  Risk score distribution:')
        print('   ', results['risk_score'].value_counts().sort_index().to_dict())